In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
df = pd.read_csv(
    '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/KNC_v12_16cM_fracgp0.7.node.csv',
    names=['Name', 'Phase', 'Degree', 'Sex', 'Yhap'],
    skiprows=1
)
# Subsets
EIA = df[df['Phase'] == 'KNC-EIA']['Degree']
DIA = df[df['Phase'] == 'KNC-DIA']['Degree']

EIA_M = df[(df['Phase'] == 'KNC-EIA') & (df['Sex'] == 'M')]['Degree']
EIA_F = df[(df['Phase'] == 'KNC-EIA') & (df['Sex'] == 'F')]['Degree']

DIA_M = df[(df['Phase'] == 'KNC-DIA') & (df['Sex'] == 'M')]['Degree']
DIA_F = df[(df['Phase'] == 'KNC-DIA') & (df['Sex'] == 'F')]['Degree']


In [ ]:
# K–S test for EIA vs DIA
ks_stat_eia_dia, p_value_eia_dia = ks_2samp(EIA, DIA)
print("EIA vs DIA:")
print(f"KS statistic = {ks_stat_eia_dia:.4f}, p-value = {p_value_eia_dia:.4g}")

# K–S test for EIA Male vs DIA Male
ks_stat_eia_dia_M, p_value_eia_dia_M = ks_2samp(EIA_M, DIA_M)
print("\nEIA Male vs DIA Male:")
print(f"KS statistic = {ks_stat_eia_dia_M:.4f}, p-value = {p_value_eia_dia_M:.4g}")

# K–S test for EIA vs DIA Female
ks_stat_eia_dia_F, p_value_eia_dia_F = ks_2samp(EIA_F, DIA_F)
print("\nEIA Female vs DIA Female:")
print(f"KS statistic = {ks_stat_eia_dia_F:.4f}, p-value = {p_value_eia_dia_F:.4g}")

# K–S test for EIA Male vs Female
ks_stat_eia_mf, p_value_eia_mf = ks_2samp(EIA_M, EIA_F)
print("\nEIA Male vs Female:")
print(f"KS statistic = {ks_stat_eia_mf:.4f}, p-value = {p_value_eia_mf:.4g}")

# K–S test for DIA Male vs Female
ks_stat_dia_mf, p_value_dia_mf = ks_2samp(DIA_M, DIA_F)
print("\nDIA Male vs Female:")
print(f"KS statistic = {ks_stat_dia_mf:.4f}, p-value = {p_value_dia_mf:.4g}")


In [ ]:
import numpy as np
import math
from scipy.stats import ks_2samp

def permutation_ks_test(data1, data2, n_perm=1000, random_state=None, group_name="Group Comparison"):
    """
    Performs permutation K-S test between two groups.
    Also calculates total number of possible unique permutations.
    """
    rng = np.random.default_rng(random_state)

    # Calculate observed statistic
    observed_stat = ks_2samp(data1, data2).statistic

    # Group sizes
    n1, n2 = len(data1), len(data2)
    total_inds = n1 + n2

    # Total unique combinations = binomial coefficient
    total_combinations = math.comb(total_inds, n1)

    # Run permutation test
    combined = np.concatenate([data1, data2])
    perm_stats = []
    for _ in range(n_perm):
        rng.shuffle(combined)
        perm1 = combined[:n1]
        perm2 = combined[n1:]
        stat = ks_2samp(perm1, perm2).statistic
        perm_stats.append(stat)

    perm_stats = np.array(perm_stats)
    p_value = (np.sum(perm_stats >= observed_stat) + 1) / (n_perm + 1)

    # Print group information
    print(f"\n--- {group_name} ---")
    print(f"Size of group 1: {n1}")
    print(f"Size of group 2: {n2}")
    print(f"Total individuals: {total_inds}")
    print(f"Total possible unique permutations: {total_combinations:,}")
    print(f"K-S statistic: {observed_stat:.4f}")
    print(f"Empirical p-value: {p_value:.4f}")

    return observed_stat, p_value, total_combinations

# -----------------------------
# Example: Run for each comparison
# -----------------------------
stat_eia_dia, p_perm_eia_dia, total_perm_eia_dia = permutation_ks_test(
    EIA.to_numpy(), DIA.to_numpy(), n_perm=1000, group_name="EIA vs DIA"
)

stat_eia_dia_M, p_perm_eia_dia_M, total_perm_eia_dia_M = permutation_ks_test(
    EIA_M.to_numpy(), DIA_M.to_numpy(), n_perm=1000, group_name="EIA Male vs DIA Male"
)

stat_eia_dia_F, p_perm_eia_dia_F, total_perm_eia_dia_F = permutation_ks_test(
    EIA_F.to_numpy(), DIA_F.to_numpy(), n_perm=1000, group_name="EIA Female vs DIA Female"
)

stat_eia_mf, p_perm_eia_mf, total_perm_eia_mf = permutation_ks_test(
    EIA_M.to_numpy(), EIA_F.to_numpy(), n_perm=1000, group_name="EIA Male vs Female"
)

stat_dia_mf, p_perm_dia_mf, total_perm_dia_mf = permutation_ks_test(
    DIA_M.to_numpy(), DIA_F.to_numpy(), n_perm=1000, group_name="DIA Male vs Female"
)


In [ ]:
import numpy as np

def permutation_ks_test(data1, data2, n_perm=1000, random_state=None):
    rng = np.random.default_rng(random_state)
    observed_stat = ks_2samp(data1, data2).statistic

    combined = np.concatenate([data1, data2])
    n1 = len(data1)

    perm_stats = []
    for _ in range(n_perm):
        rng.shuffle(combined)
        perm1 = combined[:n1]
        perm2 = combined[n1:]
        stat = ks_2samp(perm1, perm2).statistic
        perm_stats.append(stat)

    perm_stats = np.array(perm_stats)
    p_value = (np.sum(perm_stats >= observed_stat) + 1) / (n_perm + 1)

    return observed_stat, p_value

# Run permutation K–S test
stat_eia_dia, p_perm_eia_dia = permutation_ks_test(EIA.to_numpy(), DIA.to_numpy(), n_perm=1000)
stat_eia_dia_M, p_perm_eia_dia_M = permutation_ks_test(EIA_M.to_numpy(), DIA_M.to_numpy(), n_perm=1000)
stat_eia_dia_F, p_perm_eia_dia_F = permutation_ks_test(EIA_F.to_numpy(), DIA_F.to_numpy(), n_perm=1000)
stat_eia_mf, p_perm_eia_mf = permutation_ks_test(EIA_M.to_numpy(), EIA_F.to_numpy(), n_perm=1000)
stat_dia_mf, p_perm_dia_mf = permutation_ks_test(DIA_M.to_numpy(), DIA_F.to_numpy(), n_perm=1000)

print("Permutation K–S Test Results:\n")
print(f"EIA vs DIA -> KS stat: {stat_eia_dia:.4f}, Empirical p-value: {p_perm_eia_dia:.4f}")
print(f"EIA Male vs DIA Male -> KS stat: {stat_eia_dia_M:.4f}, Empirical p-value: {p_perm_eia_dia_M:.4f}")
print(f"EIA Female vs DIA Female -> KS stat: {stat_eia_dia_F:.4f}, Empirical p-value: {p_perm_eia_dia_F:.4f}")
print(f"EIA Male vs Female -> KS stat: {stat_eia_mf:.4f}, Empirical p-value: {p_perm_eia_mf:.4f}")
print(f"DIA Male vs Female -> KS stat: {stat_dia_mf:.4f}, Empirical p-value: {p_perm_dia_mf:.4f}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Load the data
# -----------------------------
df = pd.read_csv(
    '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/KNC_v12_16cM_fracgp0.7.node.csv',
    names=['Name', 'Phase', 'Degree', 'Sex', 'Yhap'],
    skiprows=1
)

# -----------------------------
# Function to compute CCDF
# -----------------------------
def compute_ccdf(degrees):
    """
    Compute complementary cumulative distribution function (CCDF)
    for degree distribution.
    """
    if len(degrees) == 0:
        return np.array([]), np.array([])
    degree_count = pd.Series(degrees).value_counts().sort_index()
    frequencies = degree_count.sort_index(ascending=False)
    cumulative_freq = np.cumsum(frequencies)
    total = len(degrees)
    P_k_greater_x = cumulative_freq / total
    return frequencies.index.to_numpy(), P_k_greater_x.to_numpy()

# -----------------------------
# Prepare data
# -----------------------------
EIA_degrees = df[df['Phase'] == 'KNC-EIA']['Degree'].to_numpy()
DIA_degrees = df[df['Phase'] == 'KNC-DIA']['Degree'].to_numpy()

# -----------------------------
# Permutation test
# -----------------------------
n_perm = 1000
rng = np.random.default_rng(seed=42)
degrees_array = df[df['Phase'].isin(['KNC-EIA', 'KNC-DIA'])]['Degree'].to_numpy()
phase_array = df[df['Phase'].isin(['KNC-EIA', 'KNC-DIA'])]['Phase'].to_numpy()

perm_lines = []

for _ in range(n_perm):
    # Shuffle phase labels
    shuffled_phase = rng.permutation(phase_array)
    
    # Select degrees labeled as DIA after shuffling
    shuffled_DIA_degrees = degrees_array[shuffled_phase == 'KNC-DIA']
    
    # Compute CCDF
    degree_values, P = compute_ccdf(shuffled_DIA_degrees)
    
    if len(degree_values) > 0:
        perm_lines.append((degree_values, P))

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8, 7), dpi=300)

# Plot permutation lines (gray)
for degree_values, P in perm_lines:
    plt.plot(degree_values, P, color='lightgray', linewidth=1.5, alpha=0.5)

# Plot observed CCDF for EIA
degree_values_EIA, P_EIA = compute_ccdf(EIA_degrees)
plt.plot(degree_values_EIA, P_EIA, color='#4CC26C', linewidth=2.5, label='EIA')

# Plot observed CCDF for DIA
degree_values_DIA, P_DIA = compute_ccdf(DIA_degrees)
plt.plot(degree_values_DIA, P_DIA, color='#365C8D', linewidth=2.5, label='DIA')

# Optional: Plot all individuals
all_degrees = np.concatenate([EIA_degrees, DIA_degrees])
degree_values_all, P_all = compute_ccdf(all_degrees)
plt.plot(degree_values_all, P_all, color='black', linewidth=3, label='All individuals')

# -----------------------------
# Final touches
# -----------------------------
plt.xlabel('Degree (k)')
plt.ylabel('P(k > x)')
#plt.title('Degree Distribution: EIA vs DIA')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Load the data
# -----------------------------
df = pd.read_csv(
    '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/KNC_v12_16cM_fracgp0.7.node.csv',
    names=['Name', 'Phase', 'Degree', 'Sex', 'Yhap'],
    skiprows=1
)

# -----------------------------
# Function to compute CCDF
# -----------------------------
def compute_ccdf(degrees):
    """
    Compute complementary cumulative distribution function (CCDF)
    for degree distribution.
    """
    if len(degrees) == 0:
        return np.array([]), np.array([])
    
    degree_count = pd.Series(degrees).value_counts().sort_index()
    frequencies = degree_count.sort_index(ascending=False)
    cumulative_freq = np.cumsum(frequencies)
    total = len(degrees)
    P_k_greater_x = cumulative_freq / total
    return frequencies.index.to_numpy(), P_k_greater_x.to_numpy()

# -----------------------------
# Prepare datasets
# -----------------------------
EIA_degrees = df[df['Phase'] == 'KNC-EIA']['Degree'].to_numpy()
DIA_degrees = df[df['Phase'] == 'KNC-DIA']['Degree'].to_numpy()

# Combined for permutation
subset_df = df[df['Phase'].isin(['KNC-EIA', 'KNC-DIA'])]
degrees_array = subset_df['Degree'].to_numpy()
phase_array = subset_df['Phase'].to_numpy()

# -----------------------------
# Permutation test
# -----------------------------
n_perm = 1000
rng = np.random.default_rng(seed=42)

perm_lines = []

for _ in range(n_perm):
    # Shuffle phase labels between EIA and DIA
    shuffled_phase = rng.permutation(phase_array)
    
    # Select shuffled degrees labeled as DIA
    shuffled_DIA_degrees = degrees_array[shuffled_phase == 'KNC-DIA']
    
    # Compute CCDF for shuffled DIA group
    degree_values, P = compute_ccdf(shuffled_DIA_degrees)
    
    if len(degree_values) > 0:
        perm_lines.append((degree_values, P))

# -----------------------------
# Plotting function
# -----------------------------
def plot_ccdf_with_permutations(EIA_degrees, DIA_degrees, perm_lines,
                                font_sizes=None, show_title=False, save_path=None):
    """
    Plot CCDF of EIA and DIA with permutation lines for null distribution.
    """
    # Default font sizes
    if font_sizes is None:
        font_sizes = {
            'title': 18,
            'axis': 16,
            'legend': 14,
            'ticks': 14
        }

    fig, ax = plt.subplots(figsize=(8, 7), dpi=600)

    # ----- Permutation null lines -----
    first_perm = True
    for degree_values, P in perm_lines:
        if first_perm:
            ax.plot(degree_values, P, color='lightgray', linewidth=1.5, alpha=0.5,
                     label=f'1,000 random permutations\nin EIA and DIA individuals')
            first_perm = False
        else:
            ax.plot(degree_values, P, color='lightgray', linewidth=1.5, alpha=0.5)

    # ----- Observed EIA -----
    degree_values_EIA, P_EIA = compute_ccdf(EIA_degrees)
    ax.plot(degree_values_EIA, P_EIA, color='#4CC26C', linewidth=2.5, label='EIA')

    # ----- Observed DIA -----
    degree_values_DIA, P_DIA = compute_ccdf(DIA_degrees)
    ax.plot(degree_values_DIA, P_DIA, color='#365C8D', linewidth=2.5, label='DIA')

    # ----- Combined (optional reference line) -----
    all_degrees = np.concatenate([EIA_degrees, DIA_degrees])
    degree_values_all, P_all = compute_ccdf(all_degrees)
    ax.plot(degree_values_all, P_all, color='black', linewidth=2.5, label='All individuals')

    # ----- Labels and styling -----
    ax.set_xlabel('Degree (k)', fontsize=font_sizes['axis'])
    ax.set_ylabel('P(k > x)', fontsize=font_sizes['axis'])
    if show_title:
        ax.set_title('Degree Distribution: EIA vs DIA', fontsize=font_sizes['title'])

    ax.tick_params(axis='both', labelsize=font_sizes['ticks'])
    ax.legend(fontsize=font_sizes['legend'])

    plt.tight_layout()
    
    # ----- Save the figure -----
    if save_path:
        fig.savefig(save_path, format='svg', bbox_inches='tight')
        print(f"Figure saved as: {save_path}")
    
    plt.show()


# -----------------------------
# Run the plot
# -----------------------------
font_sizes = {'title': 20, 'axis': 18, 'legend': 14, 'ticks': 16}
plot_ccdf_with_permutations(EIA_degrees, DIA_degrees, perm_lines, font_sizes=font_sizes, show_title=False, save_path='CDF_EIA_DIA.svg')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Load the data
# -----------------------------
df = pd.read_csv(
    '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/KNC_v12_16cM_fracgp0.7.node.csv',
    names=['Name', 'Phase', 'Degree', 'Sex', 'Yhap'],
    skiprows=1
)

# -----------------------------
# Function to compute CCDF
# -----------------------------
def compute_ccdf(degrees):
    """Compute complementary cumulative distribution function (CCDF)."""
    if len(degrees) == 0:
        return np.array([]), np.array([])
    degree_count = pd.Series(degrees).value_counts().sort_index()
    frequencies = degree_count.sort_index(ascending=False)
    cumulative_freq = np.cumsum(frequencies)
    total = len(degrees)
    P_k_greater_x = cumulative_freq / total
    return frequencies.index.to_numpy(), P_k_greater_x.to_numpy()

# -----------------------------
# Function for permutation test
# -----------------------------
def permutation_test(degrees_array, sex_array, target_label='M', n_perm=1000, rng_seed=42):
    """
    Shuffle sex labels between males and females and compute CCDFs for target group.
    """
    rng = np.random.default_rng(seed=rng_seed)
    perm_lines = []

    for _ in range(n_perm):
        shuffled_labels = rng.permutation(sex_array)
        group_degrees = degrees_array[shuffled_labels == target_label]
        degree_values, P = compute_ccdf(group_degrees)
        if len(degree_values) > 0:
            perm_lines.append((degree_values, P))
    return perm_lines

# -----------------------------
# Separate EIA and DIA datasets
# -----------------------------
EIA_df = df[df['Phase'] == 'KNC-EIA']
DIA_df = df[df['Phase'] == 'KNC-DIA']

# EIA
EIA_degrees_array = EIA_df['Degree'].to_numpy()
EIA_sex_array = EIA_df['Sex'].to_numpy()
EIA_male_degrees = EIA_df[EIA_df['Sex'] == 'M']['Degree'].to_numpy()
EIA_female_degrees = EIA_df[EIA_df['Sex'] == 'F']['Degree'].to_numpy()

# DIA
DIA_degrees_array = DIA_df['Degree'].to_numpy()
DIA_sex_array = DIA_df['Sex'].to_numpy()
DIA_male_degrees = DIA_df[DIA_df['Sex'] == 'M']['Degree'].to_numpy()
DIA_female_degrees = DIA_df[DIA_df['Sex'] == 'F']['Degree'].to_numpy()

# -----------------------------
# Run permutation tests
# -----------------------------
n_perm = 1000
perm_lines_EIA_male = permutation_test(EIA_degrees_array, EIA_sex_array, 'M', n_perm=n_perm)
perm_lines_DIA_male = permutation_test(DIA_degrees_array, DIA_sex_array, 'M', n_perm=n_perm)

# -----------------------------
# Function to plot Male vs Female with permutation lines
# -----------------------------
def plot_phase_comparison(phase_name, male_degrees, female_degrees, perm_lines, 
                          phase_color='#4CC26C', font_sizes=None, save_path=None):
    """
    Plot CCDF for male vs female, with permutation null lines.
    """
    # Default font sizes
    if font_sizes is None:
        font_sizes = {
            'title': 16,
            'axis': 14,
            'legend': 12,
            'ticks': 12
        }

    fig, ax = plt.subplots(figsize=(8, 7), dpi=600)

    # ----- Gray permutation lines -----
    # Plot the first line with a label for the legend
    if len(perm_lines) > 0:
        first_line = True
        for degree_values, P in perm_lines:
            if first_line:
                ax.plot(degree_values, P, color='lightgray', linewidth=1.5, alpha=0.5,
                         label=f'1,000 random permutations in \n{phase_name} male and female individuals')
                first_line = False
            else:
                ax.plot(degree_values, P, color='lightgray', linewidth=1.5, alpha=0.5)

    # ----- Observed Female -----
    degree_values_F, P_F = compute_ccdf(female_degrees)
    ax.plot(degree_values_F, P_F, color=phase_color, linewidth=2.5, label=f'{phase_name} Female')

    # ----- Observed Male -----
    degree_values_M, P_M = compute_ccdf(male_degrees)
    ax.plot(degree_values_M, P_M, color=phase_color, linewidth=2.5, linestyle='--', 
             label=f'{phase_name} Male')

    # ----- Labels and styling -----
    ax.set_xlabel('Degree (k)', fontsize=font_sizes['axis'])
    ax.set_ylabel('P(k > x)', fontsize=font_sizes['axis'])
    #plt.title(f'Degree Distribution: {phase_name} Male vs Female', fontsize=font_sizes['title'])

    ax.tick_params(axis='both', labelsize=font_sizes['ticks'])
    ax.legend(fontsize=font_sizes['legend'])

    plt.tight_layout()
    
    # ----- Save the figure -----
    if save_path:
        fig.savefig(f"CDF_{phase_name}_Sex.svg", format='svg', bbox_inches='tight')
        print(f"Figure saved as: {save_path}")

    plt.show()

# -----------------------------
# Plot EIA and DIA
# -----------------------------
font_sizes = {'title': 18, 'axis': 18, 'legend': 14, 'ticks': 16}

plot_phase_comparison('EIA', EIA_male_degrees, EIA_female_degrees, 
                      perm_lines_EIA_male, phase_color='#4CC26C', font_sizes=font_sizes,save_path="CDF_EIA_Sex.svg")

plot_phase_comparison('DIA', DIA_male_degrees, DIA_female_degrees, 
                      perm_lines_DIA_male, phase_color='#365C8D', font_sizes=font_sizes,save_path="CDF_DIA_Sex.svg")
